## 1. Imports

In [1]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(".."))

# EarlyFusion models
from EarlyFusion.EarlyFusionMLP import *
from EarlyFusion.EarlyFusionCNN import *
from EarlyFusion.utils import cross_validation_5fold_early_fusion

## 2. Data Loading

In [2]:
AUDIO_FEATURES = "/mloscratch/users/gnahas/data/features/acoustic_features_vf.csv"
LABELS = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/paired_healthcode.csv"
TRAIN_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_train.csv"
VAL_TEST_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_val_test.csv"

## 3. Concatenation

In [3]:
# # Save extracted features as CSV for CNN evaluation
# extracted_features_csv_path = "path/to/concatenated_extracted_features.csv"
# extracted_features_df = pd.DataFrame(extracted_features)
# # extracted_features_df.to_csv(extracted_features_csv_path, index=False)

# print(f"Extracted features saved to: {extracted_features_csv_path}")
# print(f"Extracted features shape: {extracted_features_df.shape}")

## 4. Cross-Validation

In [3]:
# Load the data files
features_df = pd.read_csv(AUDIO_FEATURES)
labels_df = pd.read_csv(LABELS)
train_folds_df = pd.read_csv(TRAIN_FOLDS)
val_test_folds_df = pd.read_csv(VAL_TEST_FOLDS)

print(f"Features shape: {features_df.shape}")
print(f"Labels shape: {labels_df.shape}")
print(f"Train folds shape: {train_folds_df.shape}")
print(f"Val/Test folds shape: {val_test_folds_df.shape}")

# Extract features (skip first 3 columns which are patient IDs)
feature_columns = features_df.columns[3:]
features = features_df[feature_columns].values

print(f"\nFeature matrix shape (after removing ID columns): {features.shape}")

# Get healthcodes
features_healthcodes = features_df['healthcode'].values 

# Create mappings
healthcode_to_idx = {hc: idx for idx, hc in enumerate(features_healthcodes)}
label_map = {hc: label for hc, label in zip(labels_df['healthCode'].values, labels_df['label_PD'].values)}

# Get data for fold 0 as example
fold = 0
train_fold_mask = (train_folds_df['fold_iteration'] == fold) & (train_folds_df['subset'] == 'train')
train_healthcodes = train_folds_df[train_fold_mask]['healthCode'].values

test_fold_mask = (val_test_folds_df['fold_iteration'] == fold) & (val_test_folds_df['subset'] == 'test')
test_healthcodes = val_test_folds_df[test_fold_mask]['healthCode'].values

# Filter valid healthcodes
train_valid_hc = [hc for hc in train_healthcodes if hc in healthcode_to_idx and hc in label_map]
test_valid_hc = [hc for hc in test_healthcodes if hc in healthcode_to_idx and hc in label_map]

# Get indices and create datasets
train_idx = np.array([healthcode_to_idx[hc] for hc in train_valid_hc], dtype=np.int64)
test_idx = np.array([healthcode_to_idx[hc] for hc in test_valid_hc], dtype=np.int64)

X_train = features[train_idx]
y_train = np.array([label_map[hc] for hc in train_valid_hc], dtype=np.float32)

X_test = features[test_idx]
y_test = np.array([label_map[hc] for hc in test_valid_hc], dtype=np.float32)

print(f"\nFold {fold} data:")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"Train label distribution: {np.bincount(y_train.astype(int))}")
print(f"Test label distribution: {np.bincount(y_test.astype(int))}")

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Initialize model
model = EarlyFusionMLP(
    input_dim=X_train.shape[1],
    hidden_dims=[512, 256, 128],
    dropout=0.5,
    verbose=True
)

# Train model
print("\n" + "="*50)
print("Training EarlyFusionMLP...")
print("="*50)
model.fit(
    X_train_tensor, 
    y_train_tensor,
    epochs=50,
    batch_size=64,
    lr=0.001,
    weight_decay=0.0,
    verbose=True
)

# Test model
print("\n" + "="*50)
print("Testing EarlyFusionMLP...")
print("="*50)
test_metrics = model.test(X_test_tensor, y_test_tensor)

print(f"Test Loss: {test_metrics['test_loss']:.4f}")
print(f"Test Accuracy: {test_metrics['test_acc']:.4f}")
print(f"Test F1 Score: {test_metrics['test_f1']:.4f}")
print(f"Test ROC AUC: {test_metrics['test_roc_auc']:.4f}")
print(f"\nConfusion Matrix:\n{test_metrics['test_conf_mat']}")

Features shape: (14948, 157)
Labels shape: (532, 3)
Train folds shape: (1380, 3)
Val/Test folds shape: (1064, 3)

Feature matrix shape (after removing ID columns): (14948, 154)

Fold 0 data:
Train: X=(276, 154), y=(276,)
Test:  X=(107, 154), y=(107,)
Train label distribution: [138 138]
Test label distribution: [46 61]
Input dimension: 154
MLP architecture: 154 -> 512 -> 256 -> 128 -> 1 (binary classification)
Device: cuda

Training EarlyFusionMLP...


/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/EarlyFusion/EarlyFusionMLP.py:135: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/Scalar.cpp:22.)
  epoch_loss += loss.item() * len(lbl)


Epoch 10/50 | Loss: 0.7086 | Acc: 0.5543
Epoch 20/50 | Loss: 0.6829 | Acc: 0.6413
Epoch 30/50 | Loss: 0.6376 | Acc: 0.6268
Epoch 40/50 | Loss: 0.5976 | Acc: 0.6594
Epoch 50/50 | Loss: 0.5736 | Acc: 0.6884
Training complete. Final - Loss: 0.5736, Acc: 0.6884

Testing EarlyFusionMLP...
Test Loss: 0.6967
Test Accuracy: 0.6168
Test F1 Score: 0.6239
Test ROC AUC: 0.6458

Confusion Matrix:
[[32 14]
 [27 34]]


In [4]:
output_dir_mlp = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/models/EarlyFusion"

print("Running 5-fold CV with EarlyFusionMLP on Extracted Features...")
best_model_path_mlp_features, metrics_history_mlp_features = cross_validation_5fold_early_fusion(
    features_csv=AUDIO_FEATURES,
    labels_csv=LABELS,
    train_folds_csv=TRAIN_FOLDS,
    val_test_folds_csv=VAL_TEST_FOLDS,
    output_dir=output_dir_mlp,
    NN_type="MLP",
    hidden_dims=[512, 256, 128],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    verbose=True
)
print(f"Best MLP model (features) saved at: {best_model_path_mlp_features}")

Running 5-fold CV with EarlyFusionMLP on Extracted Features...
5-Fold Cross-Validation with MLP
Input dimension: 154
Hidden dimensions: [512, 256, 128]
Number of samples: 14948
Device: cuda

Fold 1/5
Train samples: 276, Test samples: 107
Input dimension: 154
MLP architecture: 154 -> 512 -> 256 -> 128 -> 1 (binary classification)
Device: cuda
Training MLP model...


TypeError: 'int' object is not callable

## 5. Results Visualization

In [ ]:
# Prepare data for MLP (features) visualization
mlp_folds = metrics_history_mlp_features['folds']
mlp_acc = metrics_history_mlp_features['test_acc']
mlp_f1 = metrics_history_mlp_features['test_f1']
mlp_auc = metrics_history_mlp_features['test_roc_auc']
mlp_loss = metrics_history_mlp_features['test_loss']

print("MLP Results (Extracted Features):")
print(f"  Accuracy: {np.mean(mlp_acc):.4f} ± {np.std(mlp_acc):.4f}")
print(f"  F1 Score: {np.mean(mlp_f1):.4f} ± {np.std(mlp_f1):.4f}")
print(f"  ROC AUC:  {np.mean(mlp_auc):.4f} ± {np.std(mlp_auc):.4f}")
print(f"  Loss:     {np.mean(mlp_loss):.4f} ± {np.std(mlp_loss):.4f}")

In [ ]:
# Create MLP (features) comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-Validation Results: MLP on Extracted Features', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
axes[0, 0].plot(mlp_folds, mlp_acc, marker='s', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].axhline(np.mean(mlp_acc), color='steelblue', linestyle='--', alpha=0.5)
axes[0, 0].fill_between(mlp_folds, np.mean(mlp_acc) - np.std(mlp_acc), 
                         np.mean(mlp_acc) + np.std(mlp_acc), alpha=0.2, color='steelblue')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Test Accuracy per Fold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(mlp_folds, mlp_f1, marker='s', linewidth=2, markersize=8, color='darkgreen')
axes[0, 1].axhline(np.mean(mlp_f1), color='darkgreen', linestyle='--', alpha=0.5)
axes[0, 1].fill_between(mlp_folds, np.mean(mlp_f1) - np.std(mlp_f1), 
                         np.mean(mlp_f1) + np.std(mlp_f1), alpha=0.2, color='darkgreen')
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Test F1 Score per Fold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: ROC AUC
axes[1, 0].plot(mlp_folds, mlp_auc, marker='s', linewidth=2, markersize=8, color='darkorange')
axes[1, 0].axhline(np.mean(mlp_auc), color='darkorange', linestyle='--', alpha=0.5)
axes[1, 0].fill_between(mlp_folds, np.mean(mlp_auc) - np.std(mlp_auc), 
                         np.mean(mlp_auc) + np.std(mlp_auc), alpha=0.2, color='darkorange')
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('ROC AUC')
axes[1, 0].set_title('Test ROC AUC per Fold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loss
axes[1, 1].plot(mlp_folds, mlp_loss, marker='s', linewidth=2, markersize=8, color='crimson')
axes[1, 1].axhline(np.mean(mlp_loss), color='crimson', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(mlp_folds, np.mean(mlp_loss) - np.std(mlp_loss), 
                         np.mean(mlp_loss) + np.std(mlp_loss), alpha=0.2, color='crimson')
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Test Loss per Fold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()